# 🏆 BIST KESİN SİNYAL — ENGULFING çekirdeği + Ichimoku teyitli GÜÇLÜ AL

Bu notebook, art arda yapılan **dürüst backtest**lerin vardığı **kesin sonucu** canlı bir motora çevirir.
16 stratejiyi rastgeleye + maliyete + OOS'a (örneklem-dışı) + cooldown'a karşı elediğimizde, geriye
işlem edilebilir **tek bir gerçek edge** kaldı: **ENGULFING** — ve onun Ichimoku ile teyitlenmiş güçlü hali.

### Kesin sonuç (maliyet + OOS + cooldown üçünü birden geçen)
| Ürün | net_R | excess_R | t | OOS_exc | Hüküm |
|---|---|---|---|---|---|
| **AL (ENGULFING)** | +0.003 | +0.051 | **2.97** | **+0.060** | ✅ GEÇTİ — edge GERÇEK, OOS'ta daha güçlü. Ama net ≈ başabaş (maliyete duyarlı) |
| **GÜÇLÜ AL (ENG+ICHI)** | **+0.052** | +0.100 | 1.67 | +0.023 | 🟢 NET para burada (17×); t<2 sadece örneklem (~620 işlem), yön in-sample+OOS hep + |
| ICHIMOKU tek | −0.045 | +0.002 | 0.10 | −0.069 | ❌ sadece teyit — tek başına edge yok |
| DERIN_DEGER | −0.135 | −0.087 | −4.44 | −0.039 | ⛔ intraday'de sağlam zararlı → atıldı |

### Bu motorun mantığı (3 satır)
1. **Edge kanıtlandı ama mesele MALİYET.** ENGULFING net'i düşük komisyonda kâra geçer, yüksekte başabaşa
   düşer. Asıl kâr **teyitli üründe**: ENG+ICHI = **GÜÇLÜ AL** (net +0.052R).
2. **`KOMISYON_BPS`'i kendi GERÇEK maliyetine ayarla.** Düşükse düz AL da kârlı (`SADECE_GUCLU=False`);
   yüksekse yalnız GÜÇLÜ AL işle (`SADECE_GUCLU=True`).
3. **Her çalıştırmada kendini yeniden kanıtlar:** net_R + excess (rastgeleye karşı) + OOS'u taze veride
   ölçer, hüküm verir. Sayılar tutmazsa sinyal üretmez.

> **Veri (kilitli):** OHLCV → tvDatafeed · Evren → tradingview-screener `.limit(2000)` · yfinance YOK.
> Günlük barlarla tarar; giriş bir sonraki seans açılışında. Karar-destek aracıdır, yatırım tavsiyesi değildir.

In [ ]:
# === HÜCRE 1: KURULUM ===
import subprocess,sys
def _pip(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p],check=False)
_pip("--upgrade","git+https://github.com/rongardF/tvdatafeed.git")
_pip("tradingview-screener","pandas","numpy","matplotlib","openpyxl")
import os,gc,warnings,numpy as np,pandas as pd,matplotlib.pyplot as plt
warnings.filterwarnings("ignore"); from datetime import datetime
plt.rcParams.update({"figure.facecolor":"#0d1117","axes.facecolor":"#0d1117","savefig.facecolor":"#0d1117",
  "text.color":"#e6edf3","axes.labelcolor":"#e6edf3","xtick.color":"#8b949e","ytick.color":"#8b949e",
  "axes.edgecolor":"#30363d","grid.color":"#21262d"})
try:
    from google.colab import drive; drive.mount("/content/drive"); BASE="/content/drive/MyDrive/BIST_Backtest"
except Exception: BASE="./BIST_Backtest"
os.makedirs(BASE,exist_ok=True); print("Klasör:",BASE)

In [ ]:
# === HÜCRE 2: AYARLAR — kârın anahtarı burada (maliyet + SADECE_GUCLU) ===
CFG=dict(
  N_BARS=750, MIN_FIYAT=1.0, MIN_LIKIT_MTL=10,
  ATR_LEN=14, ATR_STOP=1.0, T1_R=1.5, T2_R=3.0, SURE=10,   # çıkış: STOP=1ATR · +1.5R yarı+başabaş · +3R · 10g
  # ⚠️ EN ÖNEMLİ PARAMETRE — kendi GERÇEK maliyetine ayarla (komisyon + slipaj, TEK YÖN, baz puan):
  KOMISYON_BPS=10.0,     # 10 bps = %0.10 tek yön. Düşür → düz AL kârlı; yükselt → yalnız GÜÇLÜ AL.
  # ⚙️ ÜRÜN SEÇİMİ:
  SADECE_GUCLU=False,    # True → SADECE GÜÇLÜ AL (ENG+ICHI) işlenir/gösterilir. Yüksek maliyette True yap.
  # 💰 RİSK YÖNETİMİ:
  SERMAYE=100_000.0, RISK_PCT=0.01, MAX_ESZAMANLI=8,
  UNIVERSE_LIMIT=2000,
)
CFG["MALIYET"]=CFG["KOMISYON_BPS"]/10000.0   # tek yön oransal (motor round-trip için ×2 alır)
print("Ürün: AL=ENGULFING · GÜÇLÜ AL=ENG+ICHI (Ichimoku teyitli)")
print("Maliyet (round-trip): %{:.2f}  ·  SADECE_GUCLU={}  ·  risk/işlem %{:.0f}"
      .format(CFG["MALIYET"]*2*100, CFG["SADECE_GUCLU"], CFG["RISK_PCT"]*100))
if CFG["SADECE_GUCLU"]: print("→ Yalnız GÜÇLÜ AL işlenecek (net para ürünü).")
else: print("→ Hem AL hem GÜÇLÜ AL. Maliyet yüksekse SADECE_GUCLU=True yapmayı düşün.")

In [ ]:
# === HÜCRE 3: VERİ (tvDatafeed + tradingview-screener) ===
from tvDatafeed import TvDatafeed,Interval
tv=TvDatafeed()   # temiz veri için TvDatafeed(kullanici,sifre)
def evren_getir():
    from tradingview_screener import Query
    _,df=(Query().set_markets("turkey").select("name","close","volume","average_volume_10d_calc")
          .limit(CFG["UNIVERSE_LIMIT"]).get_scanner_data())
    df=df.dropna(subset=["name"]).copy()
    df["likit_mTL"]=df["close"]*df["average_volume_10d_calc"]/1e6
    df=df[(df["close"]>=CFG["MIN_FIYAT"])&(df["likit_mTL"]>=CFG["MIN_LIKIT_MTL"])]
    s=sorted(df["name"].astype(str).unique()); print(f"Evren: {len(s)} hisse"); return s
def veri_indir(semboller):
    veri={}; hata=0
    for k,s in enumerate(semboller,1):
        if k%50==0: print(f"  {k}/{len(semboller)} indirildi ({len(veri)} ok)")
        try:
            df=tv.get_hist(s,exchange="BIST",interval=Interval.in_daily,n_bars=CFG["N_BARS"])
            if df is not None and len(df)>=150:
                veri[s]=df.rename(columns=str.lower)[["open","high","low","close","volume"]]
        except Exception: hata+=1
    print(f"İndirildi: {len(veri)} hisse ({hata} hata)"); return veri
semboller=evren_getir(); veri=veri_indir(semboller)

In [ ]:
# === HÜCRE 4: GÖSTERGELER (nedensel — geleceğe bakmaz) ===
def ema(s,n): return s.ewm(span=n,adjust=False).mean()
def atr(df,n=14):
    h,l,c=df["high"],df["low"],df["close"]; pc=c.shift()
    return pd.concat([h-l,(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1).ewm(alpha=1/n,adjust=False).mean()
def boga_engulfing(df):
    po,pc=df["open"].shift(),df["close"].shift()
    return (pc<po)&(df["close"]>df["open"])&(df["close"]>=po)&(df["open"]<=pc)
def _edge(cond): return cond & (~cond.shift(1).fillna(False))   # yükselen kenar = ilk gün gir
def ichimoku_bull(df):
    # Ichimoku BULUT REJİMİ (teyit): fiyat bulutun üstünde VE tenkan>=kijun. (Tek başına edge değil, teyit.)
    h,l,c=df["high"],df["low"],df["close"]
    ten=(h.rolling(9).max()+l.rolling(9).min())/2; kij=(h.rolling(26).max()+l.rolling(26).min())/2
    a=((ten+kij)/2).shift(26); b=((h.rolling(52).max()+l.rolling(52).min())/2).shift(26)
    return (c>np.maximum(a,b))&(ten>=kij)

In [ ]:
# === HÜCRE 5: ÜRÜNLER — AL ve GÜÇLÜ AL (kesin sonucun kodu) ===
def S_ENGULFING(df):
    c=df["close"]; return _edge(boga_engulfing(df)&(c>ema(c,20)))   # kanıtlanmış çekirdek edge
def S_AL(df):        return S_ENGULFING(df)                          # temel sinyal (maliyete duyarlı)
def S_GUCLU_AL(df):  return S_ENGULFING(df)&ichimoku_bull(df)        # Ichimoku teyitli → NET para burada
URUNLER={"AL (ENGULFING)":S_AL,"GÜÇLÜ AL (ENG+ICHI)":S_GUCLU_AL}
print("Ürünler:", " · ".join(URUNLER))
print("SADECE_GUCLU=%s → canlıda %s işlenecek."
      %(CFG["SADECE_GUCLU"], "yalnız GÜÇLÜ AL" if CFG["SADECE_GUCLU"] else "AL + GÜÇLÜ AL"))

In [ ]:
# === HÜCRE 6: BACKTEST MOTORU (yarı-çıkış+başabaş+maliyet, non-overlapping) ===
def simule_islem(df,i,atr_i):
    n=len(df)
    if i+1>=n: return None
    giris=df["open"].iloc[i+1]; R=CFG["ATR_STOP"]*atr_i
    if R<=0 or not np.isfinite(giris): return None
    stop=giris-R; t1=giris+CFG["T1_R"]*R; t2=giris+CFG["T2_R"]*R
    mR=(giris*CFG["MALIYET"]*2)/R; yari=False; sc=stop
    for j in range(i+1,min(i+1+CFG["SURE"],n)):
        hi,lo=df["high"].iloc[j],df["low"].iloc[j]
        if not yari:
            if lo<=sc: return -1.0-mR
            if hi>=t2: return 3.0-mR
            if hi>=t1: yari=True; sc=giris
        else:
            if lo<=sc: return 0.75-mR
            if hi>=t2: return 2.25-mR
    kap=df["close"].iloc[min(i+CFG["SURE"],n-1)]; rr=(kap-giris)/R
    return (0.75+0.5*rr-mR) if yari else (rr-mR)
def backtest(veri,fn,rastgele=False,seed=0,tarih_filtre=None):
    rng=np.random.default_rng(seed); R=[]; T=[]
    for s,df in veri.items():
        if len(df)<120: continue
        if tarih_filtre: df=df[(df.index>=tarih_filtre[0])&(df.index<tarih_filtre[1])]
        if len(df)<120: continue
        a=atr(df,CFG["ATR_LEN"]).values; sig=fn(df).values.astype(bool)
        if rastgele:
            ns=int(sig[60:len(df)-CFG["SURE"]].sum())
            if ns==0: continue
            ad=np.arange(60,len(df)-CFG["SURE"]); sel=rng.choice(ad,min(ns,len(ad)),replace=False)
            sig=np.zeros(len(df),bool); sig[sel]=True
        i=60
        while i<len(df)-CFG["SURE"]:
            if sig[i] and np.isfinite(a[i]):
                r=simule_islem(df,i,a[i])
                if r is not None: R.append(r); T.append(df.index[i]); i+=CFG["SURE"]; continue
            i+=1
    return pd.DataFrame({"R":R,"tarih":T})
print("Motor hazır.")

In [ ]:
# === HÜCRE 7: HER ÇALIŞTIRMADA KENDİNİ KANITLA — net_R + excess + OOS + hüküm ===
tarihler=sorted({t for df in veri.values() for t in df.index})
kes=tarihler[int(len(tarihler)*0.7)]           # son %30 = OOS (örneklem-dışı)
OOS=(kes,tarihler[-1])
rapor=[]; NET={}
for ad,fn in URUNLER.items():
    tam=backtest(veri,fn); rnd=backtest(veri,fn,rastgele=True,seed=7)
    oos=backtest(veri,fn,tarih_filtre=OOS); oosr=backtest(veri,fn,rastgele=True,seed=7,tarih_filtre=OOS)
    if tam.empty: rapor.append({"ürün":ad,"n":0}); NET[ad]=-9; continue
    r=tam["R"]; rr=rnd["R"] if not rnd.empty else pd.Series([0.0])
    net=r.mean(); excess=net-rr.mean(); t=net/(r.std()/np.sqrt(len(r))+1e-9)
    oos_exc=(oos["R"].mean()-(oosr["R"].mean() if not oosr.empty else 0.0)) if not oos.empty else np.nan
    NET[ad]=net
    gecti=(excess>0.03 and t>2 and (oos_exc is not None and oos_exc>0))
    para =(net>0 and excess>0 and (oos_exc is not None and oos_exc>0))
    if gecti:      hkm="✅ GEÇTİ (edge gerçek)"
    elif para:     hkm="🟢 NET PARA (t<2: örneklem)"
    elif excess>0: hkm="🟡 zayıf/teyit"
    else:          hkm="⛔ zararlı"
    rapor.append({"ürün":ad,"n":len(r),"win%":round((r>0).mean()*100,1),"net_R":round(net,3),
        "rastgele_R":round(rr.mean(),3),"excess_R":round(excess,3),
        "PF":round(r[r>0].sum()/(-r[r<0].sum()+1e-9),2),"t":round(t,2),
        "OOS_exc":round(oos_exc,3) if oos_exc==oos_exc else np.nan,"HÜKÜM":hkm})
rp=pd.DataFrame(rapor); print(rp.to_string(index=False))
al_net=NET.get("AL (ENGULFING)",-9); guclu_net=NET.get("GÜÇLÜ AL (ENG+ICHI)",-9)
print(f"\nMaliyet={CFG['KOMISYON_BPS']:.0f}bps → AL net={al_net:+.3f}R · GÜÇLÜ AL net={guclu_net:+.3f}R")
if al_net<=0 and not CFG["SADECE_GUCLU"]:
    print("⚠️ Bu maliyette düz AL net≈başabaş/negatif. ÖNERİ: Hücre 2'de SADECE_GUCLU=True yap (yalnız GÜÇLÜ AL).")
if guclu_net<=0:
    print("⛔ Bu maliyette GÜÇLÜ AL bile pozitif değil → komisyonun çok yüksek. İşlem yapma / maliyeti düşür.")

In [ ]:
# === HÜCRE 8: MALİYET DUYARLILIĞI — kâr komisyonun neresinde ölüyor? ===
# Maliyeti bps olarak tarayıp her noktada backtest'i yeniden koşarız (kesin, tahmin değil).
bps_list=[5,10,15,20,25,30]; al_curve=[]; g_curve=[]
_orig=CFG["MALIYET"]
for b in bps_list:
    CFG["MALIYET"]=b/10000.0
    al_curve.append(backtest(veri,S_AL)["R"].mean())
    g_curve.append(backtest(veri,S_GUCLU_AL)["R"].mean())
CFG["MALIYET"]=_orig
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(bps_list,al_curve,"o-",label="AL (ENGULFING)",color="#8b949e",lw=1.8)
ax.plot(bps_list,g_curve,"o-",label="GÜÇLÜ AL (ENG+ICHI)",color="#3fb950",lw=2.2)
ax.axhline(0,color="#f85149",lw=1,ls="--"); ax.axvline(CFG["KOMISYON_BPS"],color="#58a6ff",lw=1,ls=":",label="şu anki maliyet")
ax.set_xlabel("tek-yön maliyet (bps)"); ax.set_ylabel("net_R / işlem")
ax.set_title("Kâr komisyona duyarlı — GÜÇLÜ AL sıfır çizgisinin üstünde daha uzun dayanır")
ax.legend(facecolor="#161b22",edgecolor="#30363d"); plt.tight_layout(); plt.show()
print("OKUMA: Kendi maliyet noktanda (mavi çizgi) hangi eğri 0'ın üstünde? Sadece o ürünü işle.")
print("       GÜÇLÜ AL (yeşil) tipik olarak daha yüksek bps'e kadar kârlı kalır → maliyete dayanıklı.")

In [ ]:
# === HÜCRE 9: 🚀 CANLI SİNYAL — GÜÇLÜ AL öncelikli, maliyet-farkında, likiditeye göre sıralı ===
# Her hisse son barda: GÜÇLÜ AL mı (ENG+ICHI), yoksa düz AL mı? SADECE_GUCLU=True ise düz AL gösterilmez.
# Sıralama: önce GÜÇLÜ AL, sonra likidite (yüksek likidite = düşük slipaj = maliyet avantajı).
def _likit_mTL(df): return float((df["close"]*df["volume"]).tail(20).mean()/1e6)
def canli_sinyaller(veri):
    sat=[]
    for s,df in veri.items():
        if len(df)<120: continue
        atr_son=atr(df,CFG["ATR_LEN"]).iloc[-1]
        if not np.isfinite(atr_son) or atr_son<=0: continue
        guclu=bool(S_GUCLU_AL(df).iloc[-1]); al=bool(S_AL(df).iloc[-1])
        if guclu: tip="GÜÇLÜ AL"; bnet=guclu_net
        elif al and not CFG["SADECE_GUCLU"]: tip="AL"; bnet=al_net
        else: continue
        giris=df["close"].iloc[-1]; R=CFG["ATR_STOP"]*atr_son
        tilt=1.5 if tip=="GÜÇLÜ AL" else 1.0            # teyitli sinyale biraz daha risk (sınırlı)
        lot=int(CFG["SERMAYE"]*CFG["RISK_PCT"]*tilt/R) if R>0 else 0
        beklenen_TL=round(bnet*R*lot,0)                  # backtest net beklentisiyle işlem başına ~kâr
        sat.append({"hisse":s,"tip":tip,"son_bar":df.index[-1].date(),
            "kapanis":round(giris,2),"ATR":round(atr_son,2),"STOP":round(giris-R,2),
            "HEDEF_1.5R":round(giris+CFG["T1_R"]*R,2),"HEDEF_3R":round(giris+CFG["T2_R"]*R,2),
            "lot":lot,"risk_TL":round(lot*R,0),"pozisyon_TL":round(lot*giris,0),
            "beklenen_net_TL":beklenen_TL,"likit_mTL":round(_likit_mTL(df),1)})
    df=pd.DataFrame(sat)
    if df.empty: return df
    df["_g"]=(df["tip"]=="GÜÇLÜ AL").astype(int)
    return df.sort_values(["_g","likit_mTL"],ascending=False).drop(columns="_g").reset_index(drop=True)
sinyaller=canli_sinyaller(veri)
print("="*84)
if sinyaller.empty:
    print("Bugün ENGULFING/ENG+ICHI son barda tetiklemedi. Sinyal yok → nakitte kal.")
    kisa=pd.DataFrame()
else:
    ng=(sinyaller["tip"]=="GÜÇLÜ AL").sum()
    kisa=sinyaller.head(CFG["MAX_ESZAMANLI"])
    print(f"🚀 {len(sinyaller)} sinyal (GÜÇLÜ AL={ng}, AL={len(sinyaller)-ng}). En likit {len(kisa)} tanesi:\n")
    goster=["hisse","tip","kapanis","STOP","HEDEF_1.5R","HEDEF_3R","lot","risk_TL",
            "beklenen_net_TL","likit_mTL"]
    print(kisa[goster].to_string(index=False))
    tr=kisa["risk_TL"].sum(); bn=kisa["beklenen_net_TL"].sum()
    print(f"\nKısa liste: toplam risk {tr:,.0f} TL (%{tr/CFG['SERMAYE']*100:.1f}) · "
          f"backtest-beklenen net ≈ {bn:,.0f} TL")
    if ng: print("⭐ GÜÇLÜ AL (ENG+ICHI) var — testte net para bu üründe. Öncelik bunlarda.")
    elif CFG["SADECE_GUCLU"]: print("ℹ️ Bugün GÜÇLÜ AL yok; SADECE_GUCLU=True olduğu için düz AL gösterilmedi.")
    else: print("ℹ️ Bugün yalnız düz AL var — net≈başabaş; maliyetin düşükse işle, değilse bekle.")
print("="*84)

In [ ]:
# === HÜCRE 10: KAYDET + GÜNÜN ÖZETİ ===
stamp=datetime.now().strftime("%Y%m%d_%H%M")
if not sinyaller.empty:
    yol=os.path.join(BASE,f"kesin_sinyal_{stamp}.xlsx")
    with pd.ExcelWriter(yol) as xl:
        kisa.to_excel(xl,sheet_name="kisa_liste",index=False)
        sinyaller.to_excel(xl,sheet_name="tum_sinyaller",index=False)
        rp.to_excel(xl,sheet_name="kanit_raporu",index=False)
    print("💾 Kaydedildi:",yol)
    print(f"\nGÜNÜN ÖZETİ ({stamp}):")
    print(f"  • Evren: {len(veri)} · Maliyet: {CFG['KOMISYON_BPS']:.0f}bps · SADECE_GUCLU={CFG['SADECE_GUCLU']}")
    print(f"  • Sinyal: {len(sinyaller)} (GÜÇLÜ AL={(sinyaller['tip']=='GÜÇLÜ AL').sum()})")
    ust=sinyaller.iloc[0]; print(f"  • İlk aday: {ust['hisse']} [{ust['tip']}] likit={ust['likit_mTL']}mTL")
else:
    print("Kaydedilecek sinyal yok (bugün tetik yok).")

## 📖 Nasıl kullanmalıyım (kesin sonucun işletme kılavuzu)

**Ürünler**
- **AL = ENGULFING.** Kanıtlanmış çekirdek edge (excess +0.051R, t≈3, OOS pozitif). Ama net getiri
  komisyona çok yakın → **maliyetin düşükse** para kazandırır.
- **GÜÇLÜ AL = ENGULFING + Ichimoku teyidi.** Asıl net para burada (net ≈ +0.05R). Fiyat Ichimoku bulutunun
  üstünde ve tenkan≥kijun iken oluşan engulfing = daha temiz, maliyete daha dayanıklı sinyal.

**Tek kritik ayar — `KOMISYON_BPS` (Hücre 2):** kendi GERÇEK tek-yön maliyetini (komisyon + tipik slipaj)
gir. Hücre 8'deki eğri sana şunu söyler: senin maliyet noktanda hangi ürün sıfırın üstünde?
- Maliyet **düşük** → `SADECE_GUCLU=False` kalsın (hem AL hem GÜÇLÜ AL kârlı, daha çok fırsat).
- Maliyet **yüksek** → `SADECE_GUCLU=True` yap (yalnız GÜÇLÜ AL; düz AL'ı başabaşta işlemek anlamsız).

**İşlem kuralları (backtest ile birebir — sapma = güvenilmez sonuç)**
- **Giriş:** sinyal son barda oluştuysa bir sonraki seans açılışında al (`kapanis` yaklaşık giriştir).
- **Stop:** `STOP` (giriş − 1×ATR). Edge bu stopla ölçüldü; değiştirme.
- **Kısmi kâr:** +1.5R'de yarıyı sat, kalanın stopunu başabaşa çek. **Tam:** +3R veya 10 gün.
- **Likidite:** liste likiditeye göre sıralı — yüksek `likit_mTL` = düşük slipaj = maliyet avantajı.

**Risk yönetimi**
- İşlem başına sermayenin %1'i (`lot` otomatik). GÜÇLÜ AL'a ×1.5 sınırlı risk-tilt. Maks 8 eşzamanlı pozisyon.
- `beklenen_net_TL` = backtest net beklentisi × R × lot (kesin kâr değil, edge tahmini).

**Dürüstlük notları**
- Bu edge *canlı işlem defterinde* doğrulanana dek **yarı güvenle** kullanılmalı. Backtest net'i tahmindir.
- **Survivorship:** delist hisseler evrende yok → mutlak sayılar iyimser; excess (rastgeleye karşı fark) bunu
  büyük ölçüde nötrler. Bu yüzden mutlak değil **fark ve OOS** sütunlarına güven.
- GÜÇLÜ AL az sinyal üretebilir. Sinyal yoksa **nakitte kalmak da pozisyondur.** Karar-destektir; tavsiye değildir.